# 作业三: 使用fasttext进行数据清洗
学号：    姓名：

本次作业，我们将实战使用 fastText 进行一次完整的数据清洗过程，包括模型的训练和测试，以及使用模型进行数据清洗两个步骤
我们将对 FineWeb 的一个子数据集进行数据清洗。FineWeb 是 Hugging Face 发布的爬取互联网网页得到的一个数据集。FineWeb-Edu 是 FineWeb 数据集中有教育意义的一部分数据。本次作业我们将使用 fastText 对 FineWeb 数据集的一个子集中清洗出其中有教育意义的部分。

fastText 是一个能够快速进行文本分类的模型。其优点在于速度很快，但缺点是相对不太精准。因此目前被广泛用于对大规模语料进行文本分类和数据清洗。

本次作业涉及以下几个任务：
1.  **数据预处理**：我们需要将 Hugging Face 上的数据集修改为 fastText 可以接受的形式。
2.  **自行实现一个fasttext模型**: 我们需要利用pytorch自行实现一个fastText模型。
3.  **模型的训练与评测**：我们需要训练一个 fastText 模型，用于对 FineWeb 数据集进行分类任务。
4.  **数据清洗实战**：我们将实战使用 fastText 对 FineWeb 一个数据集的小子集进行数据清洗。

由于本次作业所涉及的数据量**很大**，我们强烈建议使用 GPU 完成本次作业。同时建议使用 Linux 环境运行代码，否则可能会遇到 fasttext 库安装问题，请自行搜索解决方案。

小贴士：如果遇到计算速度很慢、磁盘占用率高等情况请降低相应的batch size。

本次作业共15分，其中完成代码填空共12分，报告共3分。

## 任务0: 加载所需要的包

In [ ]:
!pip install pandas seaborn matplotlib fasttext datasets

In [ ]:
import re
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict
import fasttext
from datasets import load_dataset, get_dataset_infos
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import os

## 任务1: 数据预处理

### 数据集介绍

我们首先对数据集进行预处理，构建训练数据集。在 Facebook 官方提供的 fastText 实现中，数据需要以 TXT 格式进行存储，且每条数据需要占一行。

我们采用 Hugging Face 官方使用 Llama 3-70B 模型进行机器标注的数据作为我们的训练集，该数据集采样自 FineWeb 数据集，共有45万条。

* **原始数据集：** [https://huggingface.co/datasets/HuggingFaceFW/fineweb](https://huggingface.co/datasets/HuggingFaceFW/fineweb)
* **机器标注数据集：** [https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu-llama3-annotations](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu-llama3-annotations)

本次实验只选取其中一部分数据。如果你有兴趣，并且有足够的计算资源（32G以上内存和24G以上显存）可以尝试进行全量数据。

我们为你准备好了本次作业所需的所有数据集，使用jaacount登录[jbox](https://pan.sjtu.edu.cn/web/share/ba20c1a08fb19dff7f8f069f9fc33b10)进行下载，并将数据文件和运行文件调整成如下格式。

```bash
HW3
├── fineweb
│   └── sample
│       └── 10BT
│           └── 000_00000.parquet
├── fineweb-edu-llama3-annotations
│   └── data
│       ├── train-00000-of-00008.parquet
│       ├── train-00001-of-00008.parquet
│       ├── train-00002-of-00008.parquet
│       ├── train-00003-of-00008.parquet
│       ├── train-00004-of-00008.parquet
│       ├── train-00005-of-00008.parquet
│       ├── train-00006-of-00008.parquet
│       └── train-00007-of-00008.parquet
├── HW3.ipynb
```

### 数据预处理

此外，为了方便模型更高效地利用 token，我们需要对每一条数据内部进行预处理，即所有的英文单词改成小写，标点符号之间应当留出一个空格，以下为一个预处理实例：

This is a CoOKing question,isn't it?\n\nGreat!
-> this is a cooking question , isn't it ? great !

In [ ]:
def reformat_text(text:str) -> str: 
    ## TODO:(1分) 预处理数据, 你需要完成以下操作:
    ## 将每条数据里的换行符去掉
    ## 将所有字母转化为小写
    ## 将标点符号.\!?,'/()前后添加空格
    
    ## END TODO
    return text

In [ ]:
class FastTextDataProcessor:
    """
    封装了为 fastText 训练下载、处理和保存数据的逻辑。
    """
    def __init__(self, config):
        self.config = config
        print("正在初始化数据处理器...")

    def _format_and_save_split(self, dataset, file_path):
        """格式化并保存数据集的一个分割。"""
        print(f"正在向 {file_path} 保存数据...")
        with open(file_path, "w", encoding="utf-8") as f:
            for item in tqdm(dataset, desc=f"处理 {os.path.basename(file_path)}"):
                 ## TODO:(1分) 预处理数据, 你需要完成以下操作:
                ## 构建分类数据 数据格式为 __label__{score} {text}
                pass
                ## END TODO

    def prepare_data(self):
        """
        下载数据集，将其分割为训练、测试和验证集，
        并以 fastText 要求的格式保存。
        """
        print(f"正在加载数据集 '{self.config['dataset_name']}'...")
        dataset = load_dataset(self.config['dataset_name'], split="train")

        # 分割数据集
        print("正在分割数据集...")

        train_dataset = dataset.select(range(450000))
        test_dataset = dataset.select(range(450000, len(dataset)))
        print(f"数据集大小 -> 训练集: {len(train_dataset)}, 测试集: {len(test_dataset)}")

        df = train_dataset.to_pandas()
        # 绘制分数分布图
        plt.figure(figsize=(10, 6))
        sns.countplot(x='score', data=df, palette='viridis')
        plt.title('Distribution of Scores in the Training Dataset')
        plt.xlabel('Score')
        plt.ylabel('Count')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.show()
        # 保存每个分割
        self._format_and_save_split(train_dataset, self.config['train_file'])
        self._format_and_save_split(test_dataset, self.config['test_file'])

        ## 节约内存
        del dataset 
        del train_dataset
        del test_dataset
        print("数据准备完成。")

In [ ]:
processor_config = {
    "dataset_name": "./fineweb-edu-llama3-annotations",
    "train_file": "dataset/train_dataset.txt",
    "test_file": "dataset/test_dataset.txt",
}
os.makedirs("dataset", exist_ok=True)

data_processor = FastTextDataProcessor(processor_config)
data_processor.prepare_data()

## 任务2: 自行实现一个fasttext模型

在本次任务中，你需要自己实现一个fasttext模型，来替换原始的fasttext。

fasttext的具体实现详见课上的ppt。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import re
import os
import numpy as np
from tqdm import tqdm
from collections import Counter
import multiprocessing


class _FastTextModel(nn.Module):
    """
    FastText模型的核心神经网络结构。

    这个类定义了EmbeddingBag层用于计算词和n-gram的平均向量，
    以及两个线性层用于分类。
    """
    def __init__(self, vocab_size: int, num_buckets: int, hidden_size: int, embedding_dim: int, num_classes: int):
        """
        初始化FastText模型。

        Args:
            vocab_size (int): 词汇表的大小 (不包括n-gram)。
            num_buckets (int): 用于哈希n-gram的桶的数量。
            hidden_size (int): 隐藏层的大小 (fastText原版中没有，此处为扩展)。
            embedding_dim (int): 词向量的维度。
            num_classes (int): 分类的类别总数。
        """
        super(_FastTextModel, self).__init__()
        # 总的嵌入表大小等于词汇表大小加上哈希桶的数量
        total_embeddings = vocab_size + num_buckets
        self.embeddings = nn.EmbeddingBag(
            num_embeddings=total_embeddings,
            embedding_dim=embedding_dim,
            mode='mean'  # 使用平均值来聚合词和n-gram的向量
        )
        # 扩展的结构，原版fasttext直接从embedding连接到输出层
        self.A = nn.Linear(embedding_dim, hidden_size)
        self.B = nn.Linear(hidden_size, num_classes)

    def forward(self, indices, offsets):
        """
        定义模型的前向传播逻辑。

        Args:
            indices (torch.Tensor): 包含一个批次中所有文本的词和n-gram索引的扁平化张量。
            offsets (torch.Tensor): 一个张量，指示`indices`中每个序列的起始位置。

        Returns:
            torch.Tensor: 每个输入文本的分类logits。
        """
        ## TODO(1分): 实现FASTTEXT的forward函数
        ## torch.embeddings的使用方法见 https://docs.pytorch.ac.cn/docs/stable/generated/torch.nn.EmbeddingBag.html
        pass
        ## END TODO


In [ ]:
## 基于torch的数据集处理
class _PreprocessedDataset(Dataset):
    """
    一个简单的PyTorch数据集类，用于加载预处理好的数据。
    """
    def __init__(self, file_path):
        """
        初始化数据集。

        Args:
            file_path (str): .pt格式的预处理数据文件路径。
        """
        self.data = torch.load(file_path)

    def __len__(self):
        """返回数据集中的样本数量。"""
        return len(self.data)

    def __getitem__(self, idx):
        """根据索引获取一个样本。"""
        return self.data[idx]


In [ ]:
# --- 模型封装和数据处理 ---
class _ModelWrapper:
    """
    一个封装了模型、数据处理、训练和预测逻辑的辅助类。
    """
    def __init__(self, config, device):
        """
        初始化模型包装器。

        Args:
            config (dict): 包含所有超参数和设置的字典。
            device (torch.device): 运行模型的设备 (CPU或GPU)。
        """
        self.config = config
        self.device = device
        self.vocab = None
        self.word_to_ix = None
        self.label_to_ix, self.ix_to_label = {}, {}

        # 如果是训练模式，则构建标签映射
        if 'input' in config:
            self.label_to_ix, self.ix_to_label = self._get_label_info(config['input'])
            self.config['num_classes'] = len(self.label_to_ix)
        
        # 模型将在词汇表构建后被初始化
        self.model = None

    def _get_label_info(self, file_path):
        """
        从文件中扫描并提取所有唯一的标签，并创建映射。

        Args:
            file_path (str): 训练数据文件路径。

        Returns:
            tuple: (label_to_ix, ix_to_label) 两个字典，用于标签和索引的相互转换。
        """
        labels = set()
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                match = re.search(r'__label__\S+', line)
                if match:
                    labels.add(match.group(0))
        sorted_labels = sorted(list(labels))
        label_to_ix = {label: i for i, label in enumerate(sorted_labels)}
        ix_to_label = {i: label for label, i in label_to_ix.items()}
        return label_to_ix, ix_to_label

    def _build_vocab(self, file_path):
        """
        根据minCount参数扫描文件以构建词汇表。

        Args:
            file_path (str): 训练数据文件路径。
        """
        word_counts = Counter()
        print(f"Building vocabulary from {file_path}...")
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                text = re.sub(r'__label__\S+\s', '', line)
                word_counts.update(text.lower().split())
        
        # 根据minCount过滤词汇
        self.vocab = [word for word, count in word_counts.items() if count >= self.config['minCount']]
        # 为未知词添加一个特殊标记
        self.word_to_ix = {word: i for i, word in enumerate(self.vocab)}
        self.word_to_ix['<UNK>'] = len(self.vocab)
        self.num_classes = len(self.label_to_ix)
        print(f"Vocabulary size: {len(self.vocab)} (words with count >= {self.config['minCount']})")

    def _generate_ngrams(self, word: str) -> list[str]:
        """
        为一个单词生成字符级别的n-gram。

        Args:
            word (str): 输入的单词。

        Returns:
            list[str]: 生成的n-gram列表。
        """
        minn = self.config['minn']
        maxn = self.config['maxn']
        if minn == 0 or maxn == 0:
            return []
            
        extended_word = '<' + word + '>'
        ngrams = []
        for n in range(minn, maxn + 1):
            if len(extended_word) >= n:
                ngrams.extend([extended_word[i:i+n] for i in range(len(extended_word) - n + 1)])
        return ngrams

    def _get_indices_for_text(self, text: str):
        """
        将一行文本转换为其对应的索引列表 (包括词索引和n-gram哈希索引)。

        Args:
            text (str): 输入的文本行。

        Returns:
            list[int]: 包含所有词和n-gram索引的列表。
        """
        indices = []
        tokens = text.lower().split()
        
        # 词索引
        vocab_size = len(self.vocab)
        unk_idx = self.word_to_ix['<UNK>']
        word_indices = [self.word_to_ix.get(token, unk_idx) for token in tokens]
        indices.extend(word_indices)

        # 字符n-gram索引 (通过哈希)
        # N-grams被哈希到从 vocab_size 开始的桶中
        ## TODO(2分): 构造哈希桶
        return indices
        ## END TODO


    def _preprocess_file(self, raw_path, processed_path):
        """
        预处理原始文本文件，并将其保存为torch张量格式，以加快训练速度。

        Args:
            raw_path (str): 原始文本文件路径。
            processed_path (str): .pt格式的输出文件路径。
        """
        processed_data = []
        with open(raw_path, 'r', encoding='utf-8') as f:
            # 为tqdm获取总行数
            num_lines = sum(1 for _ in f)
            f.seek(0) # 将文件指针重置到开头
            
            # 直接迭代文件对象以节省内存
            for line in tqdm(f, total=num_lines, desc=f"Processing {os.path.basename(raw_path)}"):
                line = line.strip()
                if not line: continue
                label_match = re.search(r'__label__\S+', line)
                if not label_match: continue
                label = label_match.group(0)
                text = re.sub(r'__label__\S+\s', '', line)
                
                indices = self._get_indices_for_text(text)
                processed_data.append((torch.LongTensor(indices), self.label_to_ix[label]))
        
        torch.save(processed_data, processed_path)
        print(f"Pre-processed data saved to {processed_path}")

    def _create_preprocessed_collate_fn(self):
        """
        创建一个用于DataLoader的collate_fn。
        
        该函数负责将一批可变长度的样本打包成一个批次，
        生成用于EmbeddingBag的indices和offsets。

        Returns:
            function: a collate_fn function.
        """
        def collate_fn(batch):
            indices_list, labels_list = zip(*batch)
            labels_tensor = torch.LongTensor(labels_list)
            offsets = [0] + [len(i) for i in indices_list]
            offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
            indices_tensor = torch.cat(indices_list)
            return indices_tensor, offsets, labels_tensor
        return collate_fn

    def save_model(self, path):
        """
        将模型、配置和词汇表保存到文件。

        Args:
            path (str): 模型保存路径。
        """
        torch.save({
            'config': self.config,
            'label_to_ix': self.label_to_ix,
            'vocab': self.vocab,
            'word_to_ix': self.word_to_ix,
            'model_state_dict': self.model.state_dict()
        }, path)

    def test(self, path):
        """
        在给定的测试文件上评估模型。

        Args:
            path (str): 测试文件路径。

        Returns:
            tuple: (样本总数, 准确率, 准确率) - 返回两次准确率以模仿fasttext库的P@1和R@1。
        """
        processed_path = path + ".pt"
        if not os.path.exists(processed_path):
            print(f"Pre-processing test file: {path}")
            self._preprocess_file(path, processed_path)
        test_dataset = _PreprocessedDataset(processed_path)
        test_dataloader = DataLoader(
            test_dataset, batch_size=64,
            collate_fn=self._create_preprocessed_collate_fn(),
            num_workers=self.config['thread']
        )
        self.model.eval()
        all_true, all_preds = [], []
        with torch.no_grad():
            for indices, offsets, targets in test_dataloader:
                ## TODO(1分): 编写测试函数
                pass
                ## END TODO
        
        n_examples = len(all_true)
        correct = sum(1 for t, p in zip(all_true, all_preds) if t == p)
        precision = correct / n_examples if n_examples > 0 else 0.0
        return n_examples, precision, precision

    def predict(self, texts, k=1):
        """
        对给定的文本进行预测。

        Args:
            texts (str or list[str]): 一个或多个待预测的文本。
            k (int): 返回top-k个最可能的标签。

        Returns:
            tuple: (labels, probabilities) 预测的标签和对应的概率。
        """
        if isinstance(texts, str): texts = [texts]
        self.model.eval()
        
        indices_list = [torch.LongTensor(self._get_indices_for_text(text)) for text in texts]
        
        dummy_labels = [0] * len(indices_list)
        collate_fn = self._create_preprocessed_collate_fn()
        indices_tensor, offsets, _ = collate_fn(list(zip(indices_list, dummy_labels)))
        indices_tensor, offsets = indices_tensor.to(self.device), offsets.to(self.device)

        with torch.no_grad():
            ## TODO(1分) 编写predict函数
            topk_probs, topk_indices = None, None
            ## END TODO
        labels = [[self.ix_to_label[idx.item()] for idx in row] for row in topk_indices]
        return labels, topk_probs.cpu().numpy()



In [ ]:
# --- 画图函数 ---
def _plot_loss(loss_history, save_path='training_loss.png'):
    """
    绘制并保存训练损失曲线图。

    Args:
        loss_history (list): 包含每个训练步骤损失值的列表。
        save_path (str): 图像保存路径。
    """
    plt.figure(figsize=(12, 6))
    plt.plot(loss_history, label='Training Loss')
    plt.title('Training Loss Curve')
    plt.xlabel('Training Steps')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.legend()
    plt.show()
    plt.savefig(save_path)
    print(f"Training loss curve saved to '{save_path}'")



In [ ]:
# --- 主接口类 ---
class fasttext:
    """
    一个模拟官方fasttext库API的主接口类。
    所有方法都定义为静态方法，以便于直接调用。
    """
    @staticmethod
    def train_supervised(
        input, lr=0.1, dim=100, ws=5, epoch=5, minCount=1,
        minCountLabel=0, minn=0, maxn=0, neg=5, wordNgrams=1,
        loss='softmax', bucket=2000000, thread=1, lrUpdateRate=100,
        t=0.0001, label='__label__', verbose=2, pretrainedVectors='',
        batch_size=32
    ):
        """
        训练一个有监督的分类模型。

        Args:
            input (str): 训练文件路径 (必需)。
            lr (float): 学习率。
            dim (int): 词向量维度。
            ws (int): 上下文窗口大小 (在此脚本中未使用，为保持API一致性)。
            epoch (int): 训练轮次。
            minCount (int): 词汇的最低词频。
            minCountLabel (int): 标签的最低词频 (未使用)。
            minn (int): 最小字符n-gram长度。0表示禁用。
            maxn (int): 最大字符n-gram长度。0表示禁用。
            neg (int): 负采样数量 (未使用)。
            wordNgrams (int): 词n-gram最大长度 (未使用)。
            loss (str): 损失函数。目前只支持 'softmax'。
            bucket (int): n-gram哈希桶的数量。
            thread (int): 线程数。默认为CPU核心数。
            lrUpdateRate (int): 学习率更新频率 (未使用)。
            t (float): 采样阈值 (未使用)。
            label (str): 标签前缀。
            verbose (int): 日志详细程度 (未使用)。
            pretrainedVectors (str): 预训练词向量文件路径 (未使用)。
            batch_size (int): 训练时的批处理大小。

        Returns:
            _ModelWrapper: 一个训练好的模型包装器实例。
        """
        if thread is None:
            thread = max(1, os.cpu_count())
    
        if loss != 'softmax':
            raise NotImplementedError(
                f"Loss function '{loss}' is not implemented. "
                f"Only 'softmax' is supported in this script."
            )
    
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        args_dict = locals()
        model_wrapper = _ModelWrapper(args_dict, device)
        
        model_wrapper._build_vocab(input)
        
        # 初始化模型
        model_wrapper.model = _FastTextModel(
            vocab_size=len(model_wrapper.word_to_ix),
            num_buckets=bucket,
            embedding_dim=dim,
            hidden_size=4 * dim, 
            num_classes=model_wrapper.num_classes
        ).to(device)

        # 预处理数据
        processed_path = input + ".pt"
        if os.path.exists(processed_path):
            os.remove(processed_path) # 如果参数更改，强制重新处理
        model_wrapper._preprocess_file(input, processed_path)
        
        # 创建 DataLoader
        train_dataset = _PreprocessedDataset(processed_path)
        train_dataloader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=model_wrapper._create_preprocessed_collate_fn(),
            num_workers=thread,
            pin_memory=True if device.type == 'cuda' else False
        )
        loss_history = []
        # TODO(0分) 定义损失函数和优化器
        loss_function = None
        ## END TODO
        optimizer = optim.AdamW(model_wrapper.model.parameters(), lr=lr)
        total_steps = epoch * len(train_dataloader)
        lr_lambda = lambda step: 1.0 - step / total_steps
        scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda) ## 线性学习率
        # 开始训练
        model_wrapper.model.train()
        print(f"Starting training on {device}...")
        for i in range(epoch):
            progress_bar = tqdm(train_dataloader, desc=f"Epoch {i+1}/{epoch}")
            for indices, offsets, targets in progress_bar:
                indices, offsets, targets = indices.to(device), offsets.to(device), targets.to(device)
                ## TODO(2分): 编写训练代码
                loss_val = None
                ## END TODO
                optimizer.step()
                scheduler.step()
                
                loss_history.append(loss_val.item())
                current_lr = optimizer.param_groups[0]['lr']
                progress_bar.set_postfix({'loss': f'{loss_val.item():.4f}', 'lr': f'{current_lr:.6f}'})
                
        _plot_loss(loss_history)
        print("Training complete.")
        return model_wrapper

    @staticmethod
    def load_model(path):
        """
        从文件加载一个预训练好的模型。

        Args:
            path (str): 模型文件路径。

        Returns:
            _ModelWrapper: 一个加载好的模型包装器实例。
        """
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        checkpoint = torch.load(path, map_location=device)
        config = checkpoint['config']
        
        model_wrapper = _ModelWrapper(config, device)
        model_wrapper.label_to_ix = checkpoint['label_to_ix']
        model_wrapper.ix_to_label = {i: l for l, i in model_wrapper.label_to_ix.items()}
        model_wrapper.vocab = checkpoint['vocab']
        model_wrapper.word_to_ix = checkpoint['word_to_ix']
        
        model_wrapper.config['num_classes'] = len(model_wrapper.label_to_ix)
        model_wrapper.model = _FastTextModel(
            vocab_size=len(model_wrapper.word_to_ix),
            num_buckets=config['bucket'],
            embedding_dim=config['dim'],
            hidden_size=config['dim'], # 保持与训练时一致
            num_classes=config['num_classes']
        ).to(device)
        model_wrapper.model.load_state_dict(checkpoint['model_state_dict'])
        model_wrapper.model.eval()
        
        print(f"Model loaded from {path}")
        return model_wrapper

## 任务3: 模型的训练和评测

本部分我们将利用fasttext官方`train_supervised()`函数对fasttext模型进行训练。详细训练超参需要你自行进行设定。

In [ ]:
### 你可以取消注释掉下面这行代码，比较你自己实现的fasttext和官方的fasttext之间的差别
### 注意，官方fasttext用c++编写，所以我们的代码会有一些慢
# import fasttext

In [ ]:
class FastTextClassifier:
    """
    处理 fastText 模型的训练、加载、评估和预测。
    """
    def __init__(self, config):
        self.config = config
        self.model = None

    def train(self):
        """使用训练数据训练 fastText 模型。"""
        print(f"开始训练模型... (将保存至 {self.config['model_path']})")
        self.model = fasttext.train_supervised(
            input=self.config['train_file'],
            ## TODO(0分) 请自行设定超参, 详见fasttext官方文档, 详细超参见下一个block
            
            ## END TODO
        )
        self.model.save_model(self.config['model_path'])
        print("训练完成。")

    def load_model(self):
        """从文件加载一个预训练的模型。"""
        print(f"从 {self.config['model_path']} 加载模型...")
        self.model = fasttext.load_model(self.config['model_path'])

    def evaluate(self):
        """在测试集上评估模型并显示详细报告。"""
        if not self.model:
            self.load_model()
        print(f"\n在 {self.config['test_file']} 上进行评估...")
        result = self.model.test(self.config['test_file'])
        print(f"总体结果 -> 样本数: {result[0]}, Precision@1: {result[1]:.4f}, Recall@1: {result[2]:.4f}")

        # 详细分析和混淆矩阵
        with open(self.config['test_file'], "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]
        
        texts = [re.sub(r'__label__\d+\s', '', line) for line in lines]
        true_labels = [int(re.match(r'__label__(\d+)', line).group(1)) for line in lines]
        
        pred_labels_raw, _ = self.model.predict(texts)
        predicted_labels = [int(p[0].replace('__label__', '')) for p in pred_labels_raw]

        # 混淆矩阵
        ## TODO(1分) 构造混淆矩阵
        ## 建议使用函数: pd.crosstab
        confusion_matrix = None
        ## END TODO
        print("\n--- 混淆矩阵 ---")
        print(confusion_matrix)


    def predict(self, text_batch):
        """对一批文本进行预测。"""
        if not self.model:
            raise RuntimeError("模型尚未训练或加载。请先调用 .train() 或 .load_model()。")
        
        labels, _ = self.model.predict(text_batch)
        scores = [int(label[0].replace('__label__', '')) for label in labels]
        return scores

这里是fasttext官方提供的超参，供参考
```
    input             # training file path (required)
    lr                # learning rate [0.1]
    dim               # size of word vectors [100]
    ws                # size of the context window [5]
    epoch             # number of epochs [5]
    minCount          # minimal number of word occurences [1]
    minCountLabel     # minimal number of label occurences [1]
    minn              # min length of char ngram [0]
    maxn              # max length of char ngram [0]
    neg               # number of negatives sampled [5]
    wordNgrams        # max length of word ngram [1]
    loss              # loss function {ns, hs, softmax, ova} [softmax]
    bucket            # number of buckets [2000000]
    thread            # number of threads [number of cpus]
    lrUpdateRate      # change the rate of updates for the learning rate [100]
    t                 # sampling threshold [0.0001]
    label             # label prefix ['__label__']
    verbose           # verbose [2]
    pretrainedVectors # pretrained word vectors (.vec file) for supervised learning []
```

https://github.com/facebookresearch/fastText/tree/main/python

使用fasttext的acc可以达到68%左右，我们在最后评分时不会将acc纳入主要参考标准，但请保证所有训练流程正确。我们推荐acc在65%以上

In [ ]:
model_config = {
    "model_path": "model.bin",
    "train_file": "dataset/train_dataset.txt",
    "test_file": "dataset/test_dataset.txt",
    
    "model_params": {
        ## TODO(0分) 请自行设定超参, 详见fasttext官方文档
        "epoch": 1,
        "lr": 1,
        "dim": 10,
        "batch_size" : 32,
        "wordNgrams": 3
        ## END TODO
},
}

classifier = FastTextClassifier(model_config)
classifier.train()
classifier.evaluate()

## 任务4: 数据清洗实战

在本部分中，我们将实战使用 fastText 清洗 FineWeb 数据集的一个子集，该子集共有约1490万条数据，共 10B token，本次作业我们只清洗其中的 500M token。

* **数据集见：**[https://huggingface.co/datasets/HuggingFaceFW/fineweb/viewer/sample-10BT](https://huggingface.co/datasets/HuggingFaceFW/fineweb/viewer/sample-10BT)

事实上，FineWeb 官方使用了一个 BERT 模型（我们后续会学到相关的知识）进行了数据清洗，得到了 FineWeb-Edu 数据集。

* **清洗后的数据见：**[https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu/viewer/sample-10BT](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu/viewer/sample-10BT)

本次作业我们不验证清洗后数据的正确性，你也不需要提交清洗后数据文件，但你需要自行比对，并且将你认为有价值的发现写入最后的报告中。

In [ ]:
from datasets import concatenate_datasets

class FineWebDataCleaner:
    """
    使用一个 fastText 分类器以流式方式清洗 FineWeb 数据集。
    """
    def __init__(self, config, classifier):
        self.config = config
        self.classifier = classifier
        print("\n正在初始化 FineWeb 数据清洗器...")

    
    def _collate_fn(self, batch):
        """为 DataLoader 准备的函数，用于从每个样本中提取文本。"""
        return [reformat_text(item['text']) for item in batch]

    def clean_data(self):
        """以流式处理数据集，按分数过滤并保存结果。"""
        print("以流式模式加载 FineWeb 数据集...")
        fw_stream = load_dataset(
            self.config['streaming_dataset_name'], 
            name=self.config['streaming_dataset_config'], 
            split="train", 
            streaming=False
        )

        data_loader = DataLoader(
            fw_stream,
            batch_size=self.config['batch_size'],
            num_workers=self.config['num_workers'],
            collate_fn=self._collate_fn,
            pin_memory=True
        )

        nums_saved = 0
        print(f"开始清洗数据。高质量数据将被保存到 {self.config['cleaned_output_file']}")
        with open(self.config['cleaned_output_file'], "w", encoding='utf-8') as f:
            for text_batch in tqdm(data_loader,  desc="清洗 FineWeb 数据"):
                ## TODO(2分) 过滤 预测 并写入文件 写入时请删除原数据换行符, 并以换行符来区分不同的数据. 也可以使用pd.DataFrame来保存数据
                pass
               ## END TODO
        print(f"\n清洗完成！共保存了 {nums_saved} 条高质量文档。")


你可以自行调整`batch_size`和`num_workers`来达到加速效果。调整超参时需注意，在kaggle平台上如占用内存超过30GB一段时间，系统会杀死整个任务。

In [ ]:
cleaner_config = {
    "streaming_dataset_name": "./fineweb",
    "streaming_dataset_config": None,
    "cleaned_output_file": "output/cleaned_dataset.txt",
    "quality_threshold": 2.5, # 只保存分数 >= 3 的文本
    "batch_size": 1000,
    "num_workers": 1,  # 使用一半的CPU核心
}
os.makedirs("output", exist_ok=True)
cleaner = FineWebDataCleaner(cleaner_config, classifier)

import time
start_time = time.time()
cleaner.clean_data()
end_time = time.time()
print(f"清洗耗时{end_time - start_time}秒。")

## 任务5：与其他类似模型比较（本部分不计分）

### 基于bert的清洗模型

本部分中，我们提供用bert进行数据清洗的方式，你可以自行运行，并与fasttext比较运行时间。


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch


class BertFineWebClassifier:
    """
    一个封装了 Hugging Face fineweb-edu-classifier 模型的分类器。
    """
    def __init__(self, model_name):
        print(f"正在从 '{model_name}' 加载模型...")
        if  torch.cuda.is_available() and torch.cuda.device_count() > 1:
            print(f"Found {torch.cuda.device_count()} GPUs. Using DataParallel.")
            self.device = "cuda" # 主设备设为cuda
            
            # 2. 正常加载模型（先加载到CPU或主GPU）
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModelForSequenceClassification.from_pretrained(model_name)
            
            # 3. 使用 nn.DataParallel 包装模型
            self.model = torch.nn.DataParallel(model)
            
            # 4. 将包装后的模型移动到GPU设备
            self.model.to(self.device)
            self.model.eval()
            
            self.max_length = self.model.module.config.max_position_embeddings
            print(f"Model loaded and wrapped with DataParallel. Max sequence length: {self.max_length}")
        else:
            print(f"正在从 '{model_name}' 加载模型...")
            self.device = 'cuda' if torch.cuda.is_available() else "cpu"
            print(f"模型将运行在: {self.device}")
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
            self.model.to(self.device)
            self.model.eval()

            self.max_length = self.model.config.max_position_embeddings

    def predict(self, text_batch):
        with torch.no_grad():
            inputs = self.tokenizer(
                text_batch, 
                return_tensors="pt", 
                padding="longest", 
                truncation=True,
                max_length=self.max_length
            )
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            outputs = self.model(**inputs)
            scores = outputs.logits.squeeze(-1).float().cpu().detach().numpy()
            return scores.tolist()


In [ ]:
# 如果你在下载huggingface模型时遇到网络问题，请自行尝试搜寻解决方案，这会在未来很有用。
# 常见的解决方案有：1）使用代理 2）使用国内镜像 3）手动下载模型并加载本地路径等。
MODEL_NAME = "HuggingFaceTB/fineweb-edu-classifier"

config = {
    'streaming_dataset_name': './fineweb',
    'streaming_dataset_config': None,
    'batch_size': 512,  # 根据你的 GPU 显存调整
    'num_workers': 1,
    'quality_threshold': 3,
    'cleaned_output_file': 'output/cleaned_edu_fineweb_data_bert.txt',
    'shard_index': 0
}

# 2. 初始化真实的分类器
# 如果有 GPU，会自动使用 "cuda"
classifier = BertFineWebClassifier(model_name=MODEL_NAME)

# 3. 初始化数据清洗器，并传入真实的分类器实例
cleaner = FineWebDataCleaner(config, classifier)

import time
start_time = time.time()
# 4. 开始执行清洗任务
cleaner.clean_data()
end_time = time.time()
print(f"清洗耗时{end_time - start_time}秒。")

## 任务6：完成实验报告（3分）

根据以上实验，分析对比fasttext的性能和效果。你也可以把实验过程遇到的问题以及如何解决的方案写在下面。如果你有一些额外的内容想要补充也可以在下面加入你新增的代码块。